In [9]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from fundus_data_toolkit.datamodules.classification import DDRDataModule
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
from multistyleseg.data.fundus.consts import Lesions, ALL_CLASSES
from fundus_odmac_toolkit.models.segmentation import batch_segment, postprocess_batch
import pandas as pd
import torch
from tqdm.auto import tqdm
from multiprocessing import Pool
import cv2
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
ddr_datamodule = DDRDataModule(
    data_dir="/home/clement/Documents/data/DDR-dataset/DR_grading/",
    img_size=1024,
    batch_size=16,
    num_workers=4,
)
ddr_datamodule = ddr_datamodule.setup_all()

In [13]:
ddr_datamodule.train.return_indices = True
dataloader = ddr_datamodule.test_dataloader()

In [14]:
outputs_folders = Path("test/od_mac")
output_segmentations = outputs_folders / "segmentation"
output_pickles = outputs_folders / "pickles"
outputs_folders.mkdir(exist_ok=True, parents=True)
output_segmentations.mkdir(exist_ok=True, parents=True)
output_pickles.mkdir(exist_ok=True, parents=True)
dataset = dataloader.dataset
root_img = Path(dataloader.dataset.img_root[0])

for batch in tqdm(dataloader, total=len(dataloader)):
    indices = batch["index"]
    batch_done = False
    for index in indices:
        # Check if output already exists, skip if so
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)
        output_pkl = output_pickles / f"{relative_path.stem}.pkl"
        if output_pkl.exists():
            batch_done = True
            break
    if batch_done:
        continue
    images = batch["image"].cuda()
    with torch.inference_mode():
        outputs = batch_segment(
            images, already_normalized=True, use_tta=False, compile=True
        )
    post_processed_results = postprocess_batch(outputs)
    for i, _ in enumerate(outputs):
        index = batch["index"][i]
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)

        df = pd.DataFrame(
            dict(
                image_id=[relative_path.stem],
                od=[post_processed_results["od_centers"][i]],
                od_valid=[post_processed_results["od_valid"][i]],
                macula=[post_processed_results["macula_centers"][i]],
                macula_valid=[post_processed_results["macula_valid"][i]],
            )
        )
        output_pkl = output_pickles / f"{relative_path.stem}.pkl"
        df.to_pickle(output_pkl)
        predicted_mask = (
            post_processed_results["masks"][i].cpu().numpy().astype(np.uint8)
        )
        output_mask = output_segmentations / f"{relative_path.stem}.png"
        cv2.imwrite(str(output_mask), predicted_mask)

  0%|          | 0/235 [00:00<?, ?it/s]